In [1]:
from dotenv import load_dotenv
import os
import sys
import time
import logging
import requests
from datetime import datetime, timedelta
from typing import Optional, Dict, Any

from openai import OpenAI

from gitsource import GithubRepositoryDataReader

from minsearch import Index

from rag_helper import RAGBase

In [2]:
#Load the environment variables from the .env file
#Environ variables include OPenAI API key and other configuration settings

load_dotenv()

True

In [3]:
#Initialize the OpenAI client using the API key from the environment variables
openai_client = OpenAI()

In [ ]:
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-14 02:45:27--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.4’

rag_helper.py.4     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-14 02:45:28 (11.0 MB/s) - ‘rag_helper.py.4’ saved [2134/2134]



### Load the data

In [5]:
# Pull the lesson pages

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [7]:
#Q1: How many lesson pages?
print(len(documents))

72


### Implement Search

In [8]:
# Preview the contents of one of the documnets
print(documents[0]['content'][:500])

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack


In [9]:
documents[20]['filename']

'02-vector-search/lessons/05-minsearch-vector.md'

In [10]:
index  = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [11]:
query = "How does the agentic loop keep calling the model until it stops?"
search_result = index.search(query, num_results=3)

In [12]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:
# How does the agentic loop keep calling the model until it stops?
# What's the filename of the first result?

search_result[0]['filename']



'01-agentic-rag/lessons/14-agentic-loop.md'

### Implementing RAG

In [13]:
rag_helper = RAGBase(index, llm_client=openai_client)

In [ ]:
# Q3. RAG
# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

response = rag_helper.rag(query)

print(response['text'])
print(response['usage'])

The agentic loop keeps calling the model inside a `while True` loop.

Each turn:
- it sends the full message history to the model,
- checks the response for any `function_call` items,
- runs those tools and appends the results,
- then calls the model again.

It stops when a response comes back with **no function calls**. At that point, `has_function_calls` stays `False`, and the loop `break`s.

So the stopping rule is: **no more tool calls this turn means the agent is done.**
{'input_tokens': 7036, 'output_tokens': 117, 'total_tokens': 7153}


### Chunking

In [17]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
# Q4. Chunking
# How many chunks do you get?

len(chunks)

295

In [19]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [21]:
chunked_index = index.fit(chunks)
rag_helper_chunked = RAGBase(chunked_index, llm_client=openai_client)
response = rag_helper_chunked.rag(query)

print(response['text'])
print(response['usage'])


It keeps a `while True` loop running and checks whether the model returned any `function_call` items.

- If there is a function call, the code executes it, appends the result to `messages`, and loops again.
- If there are no function calls on that turn, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls this turn means the model is done**.
{'input_tokens': 2221, 'output_tokens': 95, 'total_tokens': 2316}
